In [18]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Detect environment
if Path("/kaggle/input").exists():
    DATA_DIR = Path("/kaggle/input/ieee-fraud-detection")
else:
    DATA_DIR = Path("data/raw")

print("Using data directory:", DATA_DIR)

for file_path in DATA_DIR.iterdir():
    print(file_path)

Using data directory: data\raw
data\raw\test_identity.csv
data\raw\test_transaction.csv
data\raw\train_identity.csv
data\raw\train_transaction.csv


# Dagshub/Mlflow initialization

In [19]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops --quiet

Note: you may need to restart the kernel to use updated packages.


In [1]:
import dagshub
import mlflow
import mlflow.sklearn

dagshub.init(repo_owner='myvari', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

Accessing as myvari

Initialized MLflow to track repo "myvari/IEEE-CIS-Fraud-Detection"

Repository myvari/IEEE-CIS-Fraud-Detection initialized!

# Data Split

In [21]:
df_transaction = pd.read_csv(DATA_DIR / 'train_transaction.csv')
df_identity = pd.read_csv(DATA_DIR / 'train_identity.csv')

# df_transaction.columns = df_transaction.columns.str.replace("id_", "id-", regex=False)
# df_identity.columns = df_identity.columns.str.replace("id_", "id-", regex=False)

In [22]:
from sklearn.model_selection import train_test_split
df = pd.merge(df_transaction, df_identity, on='TransactionID', how='left')


df = df.sort_values("TransactionDT").reset_index(drop=True)

X = df.drop(columns=['isFraud'])
y = df['isFraud']

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )   

split_index = int(len(df) * 0.8) # time-aware split

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

# Data Cleaning

In [23]:
from sklearn.base import BaseEstimator, TransformerMixin

class DataCleaner(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        drop_transaction_id=True,
        missing_threshold=0.95,
        drop_constant=True,
        drop_high_cardinality_cols=["DeviceInfo"]
    ):
        self.drop_transaction_id = drop_transaction_id
        self.missing_threshold = missing_threshold
        self.drop_constant = drop_constant
        self.drop_high_cardinality_cols = drop_high_cardinality_cols

    def fit(self, X, y=None):
        X = X.copy()
        X.columns = X.columns.str.replace("id-", "id_", regex=False)

        self.columns_to_drop_ = []

        if self.drop_transaction_id and "TransactionID" in X.columns:
            self.columns_to_drop_.append("TransactionID")

        drop_high_cardinality_cols = (
            self.drop_high_cardinality_cols
            if self.drop_high_cardinality_cols is not None
            else []
        )

        # drop high cardinality columns if selected
        for col in drop_high_cardinality_cols:
            if col in X.columns:
                self.columns_to_drop_.append(col)

        # drop columns if missingness above threshold
        if self.missing_threshold is not None:
            missing_rates = X.isna().mean()
            high_missing_cols = missing_rates[missing_rates > self.missing_threshold].index.tolist()
            self.columns_to_drop_.extend(high_missing_cols)

        # drop constant columns
        if self.drop_constant:
            constant_cols = [
                col for col in X.columns
                if X[col].nunique(dropna=False) <= 1
            ]
            self.columns_to_drop_.extend(constant_cols)

        self.columns_to_drop_ = sorted(set(self.columns_to_drop_))

        return self


    def transform(self, X):
        X = X.copy()
        X.columns = X.columns.str.replace("id-", "id_", regex=False)

        X = X.replace([np.inf, -np.inf], np.nan) # handle infinite values if any

        X = X.drop(columns=self.columns_to_drop_, errors="ignore")

        return X

    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
import numpy as np

def make_tree_preprocessor():
    num_pipeline_tree = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    cat_pipeline_tree = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("ordinal", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipeline_tree, make_column_selector(dtype_include=np.number)),
            ("cat", cat_pipeline_tree, make_column_selector(dtype_include=object)),
        ],
        verbose_feature_names_out=False
    )

# Feature Engineering

In [25]:

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        add_identity_feature=True,
        add_amount_features=True,
        add_time_features=True,
        add_email_groups=True,
    ):
        self.add_identity_feature = add_identity_feature
        self.add_amount_features = add_amount_features
        self.add_time_features = add_time_features
        self.add_email_groups = add_email_groups

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X.columns = X.columns.str.replace("id-", "id_", regex=False)

        if self.add_identity_feature:
            if "id_01" in X.columns: # defensive check since some versions of the dataset have inconsistent id column naming
                X["has_identity"] = X["id_01"].notna().astype(int)
            else:
                X["has_identity"] = 0

        # transaction amount features
        if self.add_amount_features and "TransactionAmt" in X.columns:
            X["TransactionAmt_decimal"] = (X["TransactionAmt"] - np.floor(X["TransactionAmt"]))

        if self.add_time_features and "TransactionDT" in X.columns:
            seconds_in_hour = 3600
            seconds_in_day = seconds_in_hour * 24

            transaction_hour = (X["TransactionDT"] // seconds_in_hour) % 24
            transaction_day = X["TransactionDT"] // seconds_in_day
            transaction_week = transaction_day // 7

            X["transaction_day"] = transaction_day
            X["transaction_week"] = transaction_week

            # cyclic encoding for time features (better reflecting of their nature)
            day_of_month = transaction_day % 31
            X["transaction_day_of_month_sin"] = np.sin(2 * np.pi * day_of_month / 31)
            X["transaction_day_of_month_cos"] = np.cos(2 * np.pi * day_of_month / 31)

            weekday = transaction_day % 7
            X["transaction_weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
            X["transaction_weekday_cos"] = np.cos(2 * np.pi * weekday / 7)

            X["transaction_hour_sin"] = np.sin(2 * np.pi * transaction_hour / 24)
            X["transaction_hour_cos"] = np.cos(2 * np.pi * transaction_hour / 24)


        if self.add_email_groups:
            for col in ["P_emaildomain", "R_emaildomain"]:
                if col in X.columns:
                    X[f"{col}_group"] = X[col].apply(self._group_email_domain)

        X = X.replace([np.inf, -np.inf], np.nan)

        return X
    
    def _group_email_domain(self, value):
        if pd.isna(value):
            return "missing"

        value = str(value).lower()

        if "gmail" in value:
            return "gmail"
        elif "yahoo" in value:
            return "yahoo"
        elif value in ["hotmail.com", "outlook.com", "live.com", "msn.com"]:
            return "microsoft"
        elif "icloud" in value or "me.com" in value or "mac.com" in value:
            return "apple"
        elif "anonymous" in value:
            return "anonymous"
        elif "aol" in value:
            return "aol"
        else:
            return "other"
    
    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

# Feature Selection

In [26]:
class NumericCorrelationReducer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95, verbose=True):
        self.threshold = threshold
        self.verbose = verbose

    def fit(self, X, y=None):
        X_df = X.copy()

        self.feature_names_ = X_df.columns.tolist()
        self.numeric_features_ = X_df.select_dtypes(include=[np.number]).columns.tolist()
        self.non_numeric_features_ = [
            col for col in self.feature_names_
            if col not in self.numeric_features_
        ]

        if len(self.numeric_features_) <= 1:
            self.removed_features_ = []
            self.kept_features_ = self.feature_names_
            return self

        numeric_df = X_df[self.numeric_features_]

        corr_matrix = numeric_df.corr().abs()

        mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
        upper = corr_matrix.where(mask)

        to_remove = set()

        for col in upper.columns:
            correlated_features = upper[upper[col] > self.threshold].index.tolist()

            for other in correlated_features:
                col_score = corr_matrix[col].mean()
                other_score = corr_matrix[other].mean()

                if col_score > other_score:
                    to_remove.add(col)
                else:
                    to_remove.add(other)

        self.removed_features_ = sorted(to_remove)
        self.numeric_kept_features_ = [
            feature for feature in self.numeric_features_
            if feature not in self.removed_features_
        ]
        self.kept_features_ = [
            feature for feature in self.feature_names_
            if feature not in self.removed_features_
        ]

        if self.verbose:
            print(f"[[Numeric Correlation]] Removed {len(self.removed_features_)} features: {self.removed_features_}")
            print(f"[[Numeric Correlation]] Kept {len(self.kept_features_)} features: {self.numeric_kept_features_}")

        return self

    def transform(self, X):
        X_df = X.copy()
        X_df.columns = self.feature_names_

        return X_df[self.kept_features_]

    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [27]:
class InformationValueSelector(BaseEstimator, TransformerMixin): # before OHE to avoid dimensionality explosion and sparse matrix issues
    def __init__(
        self,
        threshold=0.01,
        n_bins=10,
        max_unique_for_categorical=100,
        min_bin_size=50,
        verbose=True
    ):
        self.threshold = threshold
        self.n_bins = n_bins
        self.max_unique_for_categorical = max_unique_for_categorical
        self.min_bin_size = min_bin_size
        self.verbose = verbose

    def fit(self, X, y):
        X_df = X.copy()
        y_series = pd.Series(y).reset_index(drop=True)
        X_df = X_df.reset_index(drop=True)

        self.feature_names_ = X_df.columns.tolist()
        self.iv_scores_ = {}

        for col in self.feature_names_:
            try:
                iv = self._calculate_feature_iv(X_df[col], y_series)
            except Exception:
                iv = 0.0

            self.iv_scores_[col] = iv

        self.iv_scores_df_ = (
            pd.DataFrame({
                "feature": list(self.iv_scores_.keys()),
                "iv": list(self.iv_scores_.values())
            })
            .sort_values("iv", ascending=False)
            .reset_index(drop=True)
        )

        self.selected_features_ = self.iv_scores_df_.loc[self.iv_scores_df_["iv"] >= self.threshold, "feature"].tolist()

        if len(self.selected_features_) == 0:
            best_feature = self.iv_scores_df_.iloc[0]["feature"] # select the best feature, avoid empty feature set
            self.selected_features_ = [best_feature]

        if self.verbose:
            print(f"[IV] Selected {len(self.selected_features_)} / {len(self.feature_names_)} features")
            print("[IV] Top features:")
            print(self.iv_scores_df_.head(20))

        return self

    def transform(self, X):
        X_df = X.copy()
        X_df.columns = self.feature_names_

        return X_df[self.selected_features_]

    def _calculate_feature_iv(self, feature, y):
        prepared_feature = self._prepare_feature(feature) # convert to categorical with appropriate binning

        df = pd.DataFrame({"feature": prepared_feature, "target": y})

        grouped = df.groupby("feature", dropna=False)["target"].agg(events="sum", total="count")

        grouped["non_events"] = grouped["total"] - grouped["events"]

        # -tiny bins to reduce instability
        grouped = grouped[grouped["total"] >= self.min_bin_size]

        if grouped.shape[0] <= 1:
            return 0.0

        total_events = grouped["events"].sum()
        total_non_events = grouped["non_events"].sum()

        if total_events == 0 or total_non_events == 0:
            return 0.0

        eps = 0.5 # smoothing factor

        grouped["event_dist"] = (grouped["events"] + eps) / (total_events + eps * grouped.shape[0])
        grouped["non_event_dist"] = (grouped["non_events"] + eps) / (total_non_events + eps * grouped.shape[0])

        grouped["woe"] = np.log(grouped["event_dist"] / grouped["non_event_dist"])
        grouped["iv_component"] = (grouped["event_dist"] - grouped["non_event_dist"]) * grouped["woe"]

        return grouped["iv_component"].sum()

    def _prepare_feature(self, feature):
        feature = feature.copy()

        if pd.api.types.is_numeric_dtype(feature):
            unique_count = feature.nunique(dropna=True)

            if unique_count > self.n_bins:
                try:
                    return pd.qcut(feature, q=self.n_bins, duplicates="drop").astype(str).fillna("missing")
                except Exception:
                    return feature.fillna(-999999)
            else:
                return feature.fillna(-999999)

        else:
            feature = feature.astype("object").where(feature.notna(), "missing")
            unique_count = feature.nunique(dropna=True)

            if unique_count > self.max_unique_for_categorical:
                top_values = feature.value_counts().head(self.max_unique_for_categorical).index
                feature = feature.where(feature.isin(top_values), "rare") # rare category for infrequent

            return feature.astype(str)

    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [28]:
class NearZeroVarianceReducer(BaseEstimator, TransformerMixin):
    def __init__(self, min_unique=2, max_dominant_ratio=0.995, verbose=True):
        self.min_unique = min_unique
        self.max_dominant_ratio = max_dominant_ratio
        self.verbose = verbose

    def fit(self, X, y=None):
        X_df = X.copy()
        self.feature_names_ = X_df.columns.tolist()
        self.removed_features_ = []

        for col in self.feature_names_:
            value_counts = X_df[col].value_counts(dropna=False, normalize=True)

            if len(value_counts) < self.min_unique:
                self.removed_features_.append(col)
            elif value_counts.iloc[0] >= self.max_dominant_ratio:
                self.removed_features_.append(col)

        self.kept_features_ = [
            col for col in self.feature_names_
            if col not in self.removed_features_
        ]

        if self.verbose:
            print(f"[Near-zero variance] Removed {len(self.removed_features_)} features")
            if len(self.removed_features_) <= 50:
                print(self.removed_features_)
            else:
                print(self.removed_features_[:50], "...")

        return self

    def transform(self, X):
        X_df = X.copy()
        return X_df[self.kept_features_]

    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

# Full Pipline

In [29]:
preprocessing_pipeline = Pipeline([
    ("cleaner", DataCleaner()),
    ("feature_engineer", FeatureEngineer()),
    ("near_zero_variance", NearZeroVarianceReducer(verbose=True)),
    ("preprocessor", make_tree_preprocessor()),
])

preprocessing_pipeline_iv = Pipeline([
    ("cleaner", DataCleaner()),
    ("feature_engineer", FeatureEngineer()),
    ("near_zero_variance", NearZeroVarianceReducer(verbose=True)),
    ("iv_selector", InformationValueSelector(verbose=True)),
    ("preprocessor", make_tree_preprocessor()),
])

# preprocessing_pipeline_corr = Pipeline([
#     ("cleaner", DataCleaner()),
#     ("feature_engineer", FeatureEngineer()),
#     ("near_zero_variance", NearZeroVarianceReducer(verbose=True)),
#     ("num_corr_reduce", NumericCorrelationReducer(verbose=True)),
#     ("preprocessor", make_tree_preprocessor()),
# ])

In [30]:
import mlflow
import joblib

mlflow.set_experiment("ieee_fraud_preprocessing")

if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_name="AdaBoost_Preprocessing_Baseline"):

    mlflow.log_param("cleaner_drop_transaction_id", True)
    mlflow.log_param("cleaner_missing_threshold", 0.95)
    mlflow.log_param("cleaner_drop_constant", True)
    mlflow.log_param("cleaner_drop_high_cardinality_cols", "DeviceInfo")

    mlflow.log_param("add_identity_feature", True)
    mlflow.log_param("add_amount_features", True)
    mlflow.log_param("add_time_features", True)
    mlflow.log_param("add_email_groups", True)
    mlflow.log_param("log_transforms_used", False)
    mlflow.log_param("skewness_based_features_used", False)

    mlflow.log_param("num_impute_strategy", "median")
    mlflow.log_param("num_missing_indicator", True)
    mlflow.log_param("cat_impute_strategy", "constant_missing")
    mlflow.log_param("categorical_encoder", "OrdinalEncoder")
    mlflow.log_param("ordinal_handle_unknown", "use_encoded_value")
    mlflow.log_param("ordinal_unknown_value", -1)
    mlflow.log_param("scaler", "None")

    mlflow.log_param("near_zero_variance_used", True)
    mlflow.log_param("near_zero_variance_max_dominant_ratio", 0.995)

    mlflow.log_param("iv_selector_used", False)
    mlflow.log_param("numeric_corr_reducer_used", False)

    preprocessing_pipeline.fit(X_train, y_train)

    joblib.dump(preprocessing_pipeline, "adaboost_preprocessing_pipeline_baseline.joblib")
    mlflow.log_artifact("adaboost_preprocessing_pipeline_baseline.joblib")

    preprocessing_run_id = mlflow.active_run().info.run_id
    print(f"Preprocessing run id: {preprocessing_run_id}")

[Near-zero variance] Removed 12 features
['C3', 'V107', 'V108', 'V111', 'V113', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V305']
Preprocessing run id: f878b9aea1614452b2740a3ecd0ce7d0
🏃 View run AdaBoost_Preprocessing_Baseline at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/f878b9aea1614452b2740a3ecd0ce7d0
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


In [31]:
if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_name="AdaBoost_Preprocessing_IV"):

    mlflow.log_param("cleaner_drop_transaction_id", True)
    mlflow.log_param("cleaner_missing_threshold", 0.95)
    mlflow.log_param("cleaner_drop_constant", True)
    mlflow.log_param("cleaner_drop_high_cardinality_cols", "DeviceInfo")

    mlflow.log_param("add_identity_feature", True)
    mlflow.log_param("add_amount_features", True)
    mlflow.log_param("add_time_features", True)
    mlflow.log_param("add_email_groups", True)
    mlflow.log_param("log_transforms_used", False)
    mlflow.log_param("skewness_based_features_used", False)

    mlflow.log_param("num_impute_strategy", "median")
    mlflow.log_param("num_missing_indicator", True)
    mlflow.log_param("cat_impute_strategy", "constant_missing")
    mlflow.log_param("categorical_encoder", "OrdinalEncoder")
    mlflow.log_param("ordinal_handle_unknown", "use_encoded_value")
    mlflow.log_param("ordinal_unknown_value", -1)
    mlflow.log_param("scaler", "None")

    mlflow.log_param("near_zero_variance_used", True)
    mlflow.log_param("near_zero_variance_max_dominant_ratio", 0.995)

    mlflow.log_param("iv_selector_used", True)
    mlflow.log_param("iv_selector_threshold", 0.01)
    mlflow.log_param("iv_n_bins", 10)
    mlflow.log_param("iv_max_unique_for_categorical", 100)
    mlflow.log_param("iv_min_bin_size", 50)

    mlflow.log_param("numeric_corr_reducer_used", False)

    preprocessing_pipeline_iv.fit(X_train, y_train)

    joblib.dump(preprocessing_pipeline_iv, "adaboost_preprocessing_pipeline_iv.joblib")
    mlflow.log_artifact("adaboost_preprocessing_pipeline_iv.joblib")

    adaboost_preprocessing_run_id = mlflow.active_run().info.run_id
    print(f"AdaBoost IV preprocessing run id: {adaboost_preprocessing_run_id}")

[Near-zero variance] Removed 12 features
['C3', 'V107', 'V108', 'V111', 'V113', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V305']
[IV] Selected 367 / 422 features
[IV] Top features:
   feature        iv
0     V258  0.918307
1     V257  0.894767
2     V189  0.774238
3     V187  0.762457
4     V201  0.762154
5     V188  0.751724
6     V200  0.750811
7     V199  0.749528
8     V246  0.749493
9     V244  0.749077
10    V243  0.746679
11    V190  0.742765
12    V242  0.735650
13    V186  0.716952
14    V245  0.655815
15    V259  0.641158
16    V171  0.635981
17    V229  0.635286
18    V230  0.633518
19    V218  0.617973
AdaBoost IV preprocessing run id: ab5f41de87ee4ae3961fedbd7d869a38
🏃 View run AdaBoost_Preprocessing_IV at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/ab5f41de87ee4ae3961fedbd7d869a38
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


In [32]:
# if mlflow.active_run() is not None:
#     mlflow.end_run()

# with mlflow.start_run(run_name="RandomForest_Preprocessing_Correlation"):

#     mlflow.log_param("cleaner_drop_transaction_id", True)
#     mlflow.log_param("cleaner_missing_threshold", 0.95)
#     mlflow.log_param("cleaner_drop_constant", True)
#     mlflow.log_param("cleaner_drop_high_cardinality_cols", "DeviceInfo")

#     mlflow.log_param("add_identity_feature", True)
#     mlflow.log_param("add_amount_features", True)
#     mlflow.log_param("add_time_features", True)
#     mlflow.log_param("add_email_groups", True)
#     mlflow.log_param("log_transforms_used", False)
#     mlflow.log_param("skewness_based_features_used", False)

#     mlflow.log_param("num_impute_strategy", "median")
#     mlflow.log_param("num_missing_indicator", True)
#     mlflow.log_param("cat_impute_strategy", "constant_missing")
#     mlflow.log_param("categorical_encoder", "OrdinalEncoder")
#     mlflow.log_param("ordinal_handle_unknown", "use_encoded_value")
#     mlflow.log_param("ordinal_unknown_value", -1)
#     mlflow.log_param("scaler", "None")

#     mlflow.log_param("near_zero_variance_used", True)
#     mlflow.log_param("near_zero_variance_max_dominant_ratio", 0.995)

#     mlflow.log_param("iv_selector_used", False)
#     mlflow.log_param("numeric_corr_reducer_used", True)
#     mlflow.log_param("numeric_corr_threshold", 0.98)

#     preprocessing_pipeline_corr.fit(X_train, y_train)

#     joblib.dump(preprocessing_pipeline_corr, "rf_preprocessing_pipeline_corr.joblib")
#     mlflow.log_artifact("rf_preprocessing_pipeline_corr.joblib")

#     rf_corr_preprocessing_run_id = mlflow.active_run().info.run_id
#     print(f"RF correlation preprocessing run id: {rf_corr_preprocessing_run_id}")

# AdaBoost with IV-selected features

In [33]:
X_train_transformed = preprocessing_pipeline_iv.transform(X_train)
X_valid_transformed = preprocessing_pipeline_iv.transform(X_test)

print(X_train_transformed.shape)
print(X_valid_transformed.shape)

(472432, 682)
(118108, 682)


In [35]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)
from sklearn.base import clone

mlflow.set_experiment("AdaBoost_Training")

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "neg_log_loss": "neg_log_loss",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

param_grid_adaboost = [
    {"n_estimators": 50,  "learning_rate": 0.5, "estimator_max_depth": 1, "estimator_class_weight": None},
    {"n_estimators": 100, "learning_rate": 0.5, "estimator_max_depth": 1, "estimator_class_weight": None},
    {"n_estimators": 100, "learning_rate": 0.2, "estimator_max_depth": 2, "estimator_class_weight": None},
    {"n_estimators": 100, "learning_rate": 0.2, "estimator_max_depth": 2, "estimator_class_weight": "balanced"},
]

results_iv = []

for params in param_grid_adaboost:
    run_name = (
        f"AdaBoost_FAST_IV_"
        f"depth{params['estimator_max_depth']}_"
        f"trees{params['n_estimators']}_"
        f"lr{params['learning_rate']}_"
        f"{'balanced' if params['estimator_class_weight'] else 'unbalanced'}"
    )

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("model_type", "AdaBoostClassifier")
        mlflow.set_tag("training_stage", "fast_holdout_screening")
        mlflow.set_tag("preprocessing_type", "IV_Preprocessing")
        mlflow.set_tag("preprocessing_run_id", adaboost_preprocessing_run_id)
        mlflow.set_tag("cv_used", False)

        mlflow.log_params(params)

        base_estimator = DecisionTreeClassifier(
            max_depth=params["estimator_max_depth"],
            class_weight=params["estimator_class_weight"],
            random_state=42
        )

        model = AdaBoostClassifier(
            estimator=base_estimator,
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            random_state=42
        )

        model.fit(X_train_transformed, y_train)

        valid_proba = model.predict_proba(X_valid_transformed)[:, 1]
        valid_preds = model.predict(X_valid_transformed)

        holdout_roc_auc = roc_auc_score(y_test, valid_proba)
        holdout_avg_precision = average_precision_score(y_test, valid_proba)
        holdout_log_loss = log_loss(y_test, valid_proba)
        holdout_f1 = f1_score(y_test, valid_preds, zero_division=0)
        holdout_precision = precision_score(y_test, valid_preds, zero_division=0)
        holdout_recall = recall_score(y_test, valid_preds, zero_division=0)

        mlflow.log_metric("holdout_roc_auc", holdout_roc_auc)
        mlflow.log_metric("holdout_average_precision", holdout_avg_precision)
        mlflow.log_metric("holdout_log_loss", holdout_log_loss)
        mlflow.log_metric("holdout_f1", holdout_f1)
        mlflow.log_metric("holdout_precision", holdout_precision)
        mlflow.log_metric("holdout_recall", holdout_recall)

        model_path = f"{run_name}.joblib"
        joblib.dump(model, model_path)
        mlflow.log_artifact(model_path)

        results_iv.append({
            "run_name": run_name,
            "preprocessing": "iv",
            **params,
            "holdout_roc_auc": holdout_roc_auc,
            "holdout_average_precision": holdout_avg_precision,
            "holdout_log_loss": holdout_log_loss,
            "holdout_f1": holdout_f1,
            "holdout_precision": holdout_precision,
            "holdout_recall": holdout_recall,
        })

        print(
            f"{run_name} | "
            f"AUC={holdout_roc_auc:.4f} | "
            f"AP={holdout_avg_precision:.4f} | "
            f"LogLoss={holdout_log_loss:.4f} | "
            f"F1={holdout_f1:.4f} | "
            f"Precision={holdout_precision:.4f} | "
            f"Recall={holdout_recall:.4f}"
        )

results_iv_df = pd.DataFrame(results_iv).sort_values("holdout_roc_auc", ascending=False)
results_iv_df

AdaBoost_FAST_IV_depth1_trees50_lr0.5_unbalanced | AUC=0.8401 | AP=0.2663 | LogLoss=0.2772 | F1=0.2865 | Precision=0.6046 | Recall=0.1877
🏃 View run AdaBoost_FAST_IV_depth1_trees50_lr0.5_unbalanced at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/5/runs/9094478a31ce41d4b9d890e1356d34df
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/5
AdaBoost_FAST_IV_depth1_trees100_lr0.5_unbalanced | AUC=0.8441 | AP=0.2921 | LogLoss=0.3194 | F1=0.3011 | Precision=0.6061 | Recall=0.2003
🏃 View run AdaBoost_FAST_IV_depth1_trees100_lr0.5_unbalanced at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/5/runs/0ba8f66436284d98b039ab57df2934a5
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/5
AdaBoost_FAST_IV_depth2_trees100_lr0.2_unbalanced | AUC=0.8496 | AP=0.3515 | LogLoss=0.2644 | F1=0.2984 | Precision=0.8163 | Recall=0.1826
🏃 View run AdaBoost_FAST_IV_depth2_

,run_name,preprocessing,n_estimators,learning_rate,estimator_max_depth,estimator_class_weight,holdout_roc_auc,holdout_average_precision,holdout_log_loss,holdout_f1,holdout_precision,holdout_recall
2,AdaBoost_FAST_IV_depth2_trees100_lr0.2_unbalanced,iv,100,0.2,2,None,0.849597,0.351537,0.264403,0.298411,0.816282,0.182579
1,AdaBoost_FAST_IV_depth1_trees100_lr0.5_unbalanced,iv,100,0.5,1,None,0.844141,0.292119,0.319357,0.301091,0.606106,0.200295
0,AdaBoost_FAST_IV_depth1_trees50_lr0.5_unbalanced,iv,50,0.5,1,None,0.840101,0.266269,0.277245,0.286519,0.604596,0.187746
3,AdaBoost_FAST_IV_depth2_trees100_lr0.2_balanced,iv,100,0.2,2,balanced,0.700418,0.081621,0.425705,0.198947,0.121980,0.539124


In [36]:
best_params = results_iv_df.iloc[0][[
    "n_estimators",
    "learning_rate",
    "estimator_max_depth",
    "estimator_class_weight",
]].to_dict()

if pd.isna(best_params["estimator_class_weight"]):
    best_params["estimator_class_weight"] = None

best_params

{'n_estimators': 100,
 'learning_rate': 0.2,
 'estimator_max_depth': 2,
 'estimator_class_weight': None}

In [38]:
cv = TimeSeriesSplit(n_splits=2)

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "neg_log_loss": "neg_log_loss"
}

with mlflow.start_run(run_name="AdaBoost_Finalist_TimeSeriesCV"):
    mlflow.set_tag("model_type", "AdaBoostClassifier")
    mlflow.set_tag("training_stage", "finalist_timeseries_cv")
    mlflow.set_tag("cv_used", True)
    mlflow.set_tag("cv_type", "TimeSeriesSplit")
    mlflow.set_tag("preprocessing_type", "IV_Preprocessing")
    mlflow.set_tag("preprocessing_run_id", adaboost_preprocessing_run_id)
    mlflow.set_tag("preprocessing_note","preprocessing fitted once before CV; IV selector was fitted on full training split")

    mlflow.log_params(best_params)
    mlflow.log_param("cv_n_splits", 2)

    base_estimator = DecisionTreeClassifier(
        max_depth=best_params["estimator_max_depth"],
        class_weight=best_params["estimator_class_weight"],
        random_state=42
    )

    finalist_model = AdaBoostClassifier(
        estimator=base_estimator,
        n_estimators=best_params["n_estimators"],
        learning_rate=best_params["learning_rate"],
        random_state=42
    )

    cv_results = cross_validate(
        finalist_model,
        X_train_transformed,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise"
    )

    train_auc = cv_results["train_roc_auc"].mean()
    val_auc = cv_results["test_roc_auc"].mean()

    train_ap = cv_results["train_average_precision"].mean()
    val_ap = cv_results["test_average_precision"].mean()

    train_log_loss = -cv_results["train_neg_log_loss"].mean()
    val_log_loss = -cv_results["test_neg_log_loss"].mean()

    mlflow.log_metric("cv_train_roc_auc", train_auc)
    mlflow.log_metric("cv_val_roc_auc", val_auc)
    mlflow.log_metric("cv_train_average_precision", train_ap)
    mlflow.log_metric("cv_val_average_precision", val_ap)
    mlflow.log_metric("cv_train_log_loss", train_log_loss)
    mlflow.log_metric("cv_val_log_loss", val_log_loss)
    mlflow.log_metric("roc_auc_gap", train_auc - val_auc)
    mlflow.log_metric("log_loss_gap", val_log_loss - train_log_loss)

    print("Random Forest Finalist CV Results")
    print("-" * 45)
    print(f"CV Train ROC-AUC: {train_auc:.4f}")
    print(f"CV Val ROC-AUC:   {val_auc:.4f}")
    print(f"CV Val AP:        {val_ap:.4f}")
    print(f"CV Val LogLoss:   {val_log_loss:.4f}")
    print(f"ROC-AUC Gap:      {train_auc - val_auc:.4f}")

Random Forest Finalist CV Results
---------------------------------------------
CV Train ROC-AUC: 0.8791
CV Val ROC-AUC:   0.8573
CV Val AP:        0.4315
CV Val LogLoss:   0.2723
ROC-AUC Gap:      0.0218
🏃 View run AdaBoost_Finalist_TimeSeriesCV at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/5/runs/ae99e168bf68437fbeb651196bf3680a
🧪 View experiment at: https://dagshub.com/myvari/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/5


In [39]:
base_estimator = DecisionTreeClassifier(
    max_depth=best_params["estimator_max_depth"],
    class_weight=best_params["estimator_class_weight"],
    random_state=42
)

best_model = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=best_params["n_estimators"],
    learning_rate=best_params["learning_rate"],
    random_state=42
)

final_pipeline = Pipeline([
    ("preprocessing", clone(preprocessing_pipeline_iv)),
    ("model", best_model)
])

with mlflow.start_run(run_name="AdaBoost_Final_FullPipeline"):
    mlflow.set_tag("model_type", "AdaBoostClassifier")
    mlflow.set_tag("training_stage", "final_full_pipeline")
    mlflow.set_tag("preprocessing_type", "IV_Preprocessing")
    mlflow.set_tag("main_metric", "holdout_roc_auc")
    mlflow.log_params(best_params)

    print("Fitting final full pipeline...")
    final_pipeline.fit(X_train, y_train)

    print("Generating validation predictions...")
    valid_proba = final_pipeline.predict_proba(X_test)[:, 1]
    valid_preds = final_pipeline.predict(X_test)

    holdout_roc_auc = roc_auc_score(y_test, valid_proba)
    holdout_avg_precision = average_precision_score(y_test, valid_proba)
    holdout_log_loss = log_loss(y_test, valid_proba)
    holdout_f1 = f1_score(y_test, valid_preds, zero_division=0)
    holdout_precision = precision_score(y_test, valid_preds, zero_division=0)
    holdout_recall = recall_score(y_test, valid_preds, zero_division=0)

    mlflow.log_metric("holdout_roc_auc", holdout_roc_auc)
    mlflow.log_metric("holdout_average_precision", holdout_avg_precision)
    mlflow.log_metric("holdout_log_loss", holdout_log_loss)
    mlflow.log_metric("holdout_f1", holdout_f1)
    mlflow.log_metric("holdout_precision", holdout_precision)
    mlflow.log_metric("holdout_recall", holdout_recall)

    model_path = "adaboost_final_full_pipeline.joblib"
    joblib.dump(final_pipeline, model_path)
    mlflow.log_artifact(model_path)

    print("\nFinal AdaBoost Full Pipeline Results")
    print("-" * 55)
    print(f"ROC-AUC:           {holdout_roc_auc:.4f}")
    print(f"Average Precision: {holdout_avg_precision:.4f}")
    print(f"Log Loss:          {holdout_log_loss:.4f}")
    print(f"F1 Score:          {holdout_f1:.4f}")
    print(f"Precision:         {holdout_precision:.4f}")
    print(f"Recall:            {holdout_recall:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, valid_preds))

    print("\nClassification Report:")
    print(classification_report(y_test, valid_preds, zero_division=0))

    final_run_id = mlflow.active_run().info.run_id
    print(f"\nFinal run id: {final_run_id}")

Fitting final full pipeline...
[Near-zero variance] Removed 12 features
['C3', 'V107', 'V108', 'V111', 'V113', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V305']
[IV] Selected 367 / 422 features
[IV] Top features:
   feature        iv
0     V258  0.918307
1     V257  0.894767
2     V189  0.774238
3     V187  0.762457
4     V201  0.762154
5     V188  0.751724
6     V200  0.750811
7     V199  0.749528
8     V246  0.749493
9     V244  0.749077
10    V243  0.746679
11    V190  0.742765
12    V242  0.735650
13    V186  0.716952
14    V245  0.655815
15    V259  0.641158
16    V171  0.635981
17    V229  0.635286
18    V230  0.633518
19    V218  0.617973
Generating validation predictions...

Final AdaBoost Full Pipeline Results
-------------------------------------------------------
ROC-AUC:           0.8496
Average Precision: 0.3515
Log Loss:          0.2644
F1 Score:          0.2984
Precision:         0.8163
Recall:            0.1826

Confusion Matrix:
[[113877    167]
 [  3322    742]]

In [2]:
print(mlflow.search_runs(
    experiment_names=["AdaBoost_Training"],
    order_by=["metrics.holdout_roc_auc DESC", "metrics.roc_auc_gap ASC", "metrics.holdout_log_loss ASC"]
)[['tags.mlflow.runName', 'metrics.cv_train_roc_auc', 'metrics.cv_val_roc_auc', 'metrics.roc_auc_gap', 'metrics.holdout_roc_auc', 'tags.preprocessing_type']].to_markdown(index=False))

| tags.mlflow.runName                               |   metrics.cv_train_roc_auc |   metrics.cv_val_roc_auc |   metrics.roc_auc_gap |   metrics.holdout_roc_auc | tags.preprocessing_type   |
|:--------------------------------------------------|---------------------------:|-------------------------:|----------------------:|--------------------------:|:--------------------------|
| AdaBoost_Final_FullPipeline                       |                 nan        |                nan       |           nan         |                  0.849597 | IV_Preprocessing          |
| AdaBoost_FAST_IV_depth2_trees100_lr0.2_unbalanced |                 nan        |                nan       |           nan         |                  0.849597 | IV_Preprocessing          |
| AdaBoost_FAST_IV_depth1_trees100_lr0.5_unbalanced |                 nan        |                nan       |           nan         |                  0.844141 | IV_Preprocessing          |
| AdaBoost_FAST_IV_depth1_trees50_lr0.5_unbalanced